<!-- nav -->
[← TypeScript tutorial](README.md) · [Home](../../README.md) · [2 · Answers you can trust →](02_Answers_You_Can_Trust.ipynb)

# 1 · Constraints as data

**The problem.** A shipment of vaccine must stay between 2 and 8 °C. That constraint lives in the QA team's spreadsheet,
in the Python pipeline that processes the data loggers' uploads, in the TypeScript dashboard that flags shipments, and
in the logger's firmware, which sounds an alarm. Four copies, in four languages. When the range changes for a new
product, someone updates three of them.

A constraint in code can't be stored, sent or read by another language: a TypeScript function means nothing to the
Python pipeline. mbse-expressions writes the constraint once, as an **expression**: data with a schema, like any
mbse-schemas object, that every program loads and evaluates the same way, and that translates into each of those
languages (case study 7).

This case study writes a constraint, evaluates it, checks it, and saves it. The cold chain's schemas are in
`toolkit.ts`, beside this notebook: a `Shipment` has the latest temperature its logger reported (`celsius`), every
reading (`readings`), the hours it has been in transit (`hours`), and the loggers it carries.

In [1]:
import { Evaluators, Expressions as E } from "@mbse/expressions";
import { Terms } from "@mbse/expressions/Framework";
import { JSON as SchemaJSON } from "@mbse/schemas/Framework";
import { cold_chain, show } from "./toolkit.ts";

const store = cold_chain();
const s1 = store.Shipment().id("S1").celsius(5.5).hours(30n).create();
const s2 = store.Shipment().id("S2").celsius(9.1).hours(80n).create();

## Step 1: writing a constraint

Constraints are built with **writers**. `E.variable("this")` is the variable the constraint is about, the shipment;
reading a property of its writer reads that property, and methods build operations. (`this` is reserved in TypeScript,
so the writer is called `self`; the variable is still `this`.)

In [2]:
const self = E.variable("this");
const celsius = self.celsius;
const in_range = E.literal(2.0).le(celsius).and_(celsius.le(8.0));
console.log(show(in_range));

and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0))


`show` (from the toolkit) writes the expression as function calls. `self.celsius` reads a property: `get`. `le` is
"less than or equal", and the two comparisons are joined by `and`. Both read the same `celsius` writer, so the
constraint holds one `get` term, used twice. These operation names are the **core vocabulary**
that every program evaluating expressions implements, in TypeScript and in Python alike. Writing the literal first,
`2.0 <= celsius`, gives exactly the expression the Python tutorial reads from `2.0 <= this.celsius <= 8.0`.

## Step 2: evaluating it

`Evaluators.OfAny(constraint, scope)` computes a constraint's value, with the variables in the scope bound to values:

In [3]:
console.log(Evaluators.OfAny(in_range, { this: s1 }), Evaluators.OfAny(in_range, { this: s2 }));

true false


A constraint about one value, called `this`, is common enough to have a shortcut, `Evaluators.predicate(constraint,
value)`:

In [4]:
console.log([s1, s2].map((shipment) => Evaluators.predicate(in_range, shipment)));

[ true, false ]


## Step 3: the same data, however it's written

A writer holds the expression it built; `Terms.same` compares two expressions by structure. Writing the comparison
the other way round, `celsius >= 2.0`, means the same, but isn't the same expression:

In [5]:
const by_hand = self.celsius.ge(2.0).and_(self.celsius.le(8.0));
console.log(show(by_hand));
console.log(Terms.same(in_range.data, E.literal(2.0).le(self.celsius).and_(self.celsius.le(8.0)).data),
  Terms.same(in_range.data, by_hand.data));

and(ge(get(this, 'celsius'), 2.0), le(get(this, 'celsius'), 8.0))


true false


Python can also read a constraint from a lambda (`Text.FromFunction`), since Python keeps a function's source; it builds
the same data as writers do.

## Step 4: what a constraint can say

A constraint can have several variables, and use values from your program. A value like `MAX_HOURS` becomes a
**literal** when the constraint is written: the expression doesn't refer back to your program, so it means the same
thing wherever it's loaded. Integers are `bigint`s, as they are in mbse-schemas: `72n`, not `72`, which is a float.

In [6]:
const MAX_HOURS = 72n;
const on_time = self.hours.le(MAX_HOURS);
const late_by = self.hours.add(E.variable("now")).gt(MAX_HOURS);
console.log(show(on_time));
console.log(show(late_by));
console.log(Evaluators.OfAny(late_by, { this: s1, now: 50n }));

le(get(this, 'hours'), 72)


gt(add(get(this, 'hours'), now), 72)


true


To name an intermediate value, use a **let**: the name is bound to the value inside the body, and nowhere else.

In [7]:
const temperature = E.variable("c");
console.log(show(E.let_("c", self.celsius, temperature.ge(2.0).and_(temperature.le(8.0)))));

let c = get(this, 'celsius') in and(ge(c, 2.0), le(c, 8.0))


Writers build only what the core vocabulary has. An operation outside it can still be written, with
`E.operation(name, ...arguments)`, but it's an extension that some programs won't implement. The core has no division,
because division by zero would make a constraint partial.

## Step 5: validating a constraint before it is evaluated

A constraint that arrives as data, from a file or another program, may be broken. `validate()` reports what evaluating
it would raise: variables nothing binds, wrong numbers of arguments, and, with `core: true`, operations outside the core
vocabulary.

In [8]:
console.log(late_by.data.validate({ bound: ["this", "now"], core: true }));
console.log(late_by.data.validate({ bound: ["this"] }));
console.log(E.operation("div", self.celsius, 2.0).data.validate({ bound: ["this"], core: true }));

[]


[ "argument 0: argument 1: variable 'now' is not bound" ]


[ "'div' is not a core operation" ]


`.data` is the expression a writer holds; most functions accept either.

## Step 6: saving and loading

An expression's data has a schema, so it saves like any object graph. `E.DIALECT.schema_of(constraint)` gives the schema
to save it by, and `E.Builders` builds expressions back from the text:

In [9]:
const schema = E.DIALECT.schema_of(in_range.data);
const text = SchemaJSON.ToJSON(E.Builders).Reachable(schema, in_range.data);
console.log(text);

{"root": "s0", "objects": {"s0": {"kind": "operation", "name": "and", "arguments": [{"argument": {"$ref": "s1", "$schema": "Expressions.OfOperation"}, "index": 0}, {"argument": {"$ref": "s2", "$schema": "Expressions.OfOperation"}, "index": 1}]}, "s1": {"kind": "operation", "name": "le", "arguments": [{"argument": {"$ref": "s3", "$schema": "Expressions.OfLiteral"}, "index": 0}, {"argument": {"$ref": "s4", "$schema": "Expressions.OfOperation"}, "index": 1}]}, "s2": {"kind": "operation", "name": "le", "arguments": [{"argument": {"$ref": "s4", "$schema": "Expressions.OfOperation"}, "index": 0}, {"argument": {"$ref": "s5", "$schema": "Expressions.OfLiteral"}, "index": 1}]}, "s3": {"kind": "literal", "float": 2.0}, "s4": {"kind": "operation", "name": "get", "arguments": [{"argument": {"$ref": "s6", "$schema": "Expressions.OfVariable"}, "index": 0}, {"argument": {"$ref": "s7", "$schema": "Expressions.OfLiteral"}, "index": 1}]}, "s5": {"kind": "literal", "float": 8.0}, "s6": {"kind": "variable

Each object carries a `kind`, arguments are entries with an `index`, and `get(this, 'celsius')`, which the constraint
uses twice, is written once. Nothing in the text is TypeScript: it's byte for byte the text the Python tutorial writes
(which prints it indented), and the Python pipeline reads it and evaluates the same constraint.

In [10]:
const loaded = SchemaJSON.FromJSON(E.Builders).Reachable(schema, text);
console.log(show(loaded as any), Terms.same(loaded, in_range.data));
console.log([s1, s2].map((shipment) => Evaluators.predicate(loaded, shipment)));

and(le(2.0, get(this, 'celsius')), le(get(this, 'celsius'), 8.0)) true


[ true, false ]


## Why it works this way

- **A constraint is data, not code.** Data can be stored beside what it's about, versioned, diffed, sent, and loaded by
  a program in another language. Code can only be run, and only by its own language.
- **A small core vocabulary.** Every operation in the core is pure, total and bounded, so every implementation can
  evaluate it the same way, and a solver could reason about it. What isn't in the core is an extension, reported by
  `validate({ core: true })`.
- **Values become literals.** A constraint that referred back into the program that wrote it couldn't be loaded anywhere
  else.

The next case study looks at what the constraints answer when the data is incomplete, which is most of the time.

<!-- nav -->
[← TypeScript tutorial](README.md) · [Home](../../README.md) · [2 · Answers you can trust →](02_Answers_You_Can_Trust.ipynb)